In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from ridb import RIDB

root = "/Users/rodionsa/KAUST Dropbox/Sergei Rodionov/Lab_measurements_shared/HoloEML/HoloEML281026/interferometer/"

In [ ]:
# SiO2 T data
with h5py.File(root+"HoloEML281026_SiO2_T.h5", 'r') as f:
    print(list(f.keys()))           # all datasets in the file
    print(dict(f.attrs))            # source folder, pipeline config, conventions...
    T_SiO2 = f["T"][:]
    T_SiO2_std = f["T_std"][:]
    wvl = f["wavelength"][:]
    
T_SiO2 = np.flip(T_SiO2, axis=2)
T_SiO2_std = np.flip(T_SiO2_std, axis=2)

In [ ]:
# SiO2 phi data

with h5py.File(root+"HoloEML281026_SiO2_aia_phi.h5", 'r') as f:
    print(list(f.keys()))           # all datasets in the file
    print(dict(f.attrs))            # source folder, pipeline config, conventions...
   
    #wvl = f["wavelength"][:]
    phi_SiO2 = f["phi"][:]
    phi_SiO2_std = f["phi_error"][:]

phi_SiO2 = np.flip(phi_SiO2, axis=2)
phi_SiO2_std = np.flip(phi_SiO2_std, axis=2)

In [ ]:
T_SiO2_rel = T_SiO2_std/T_SiO2

In [ ]:
from matplotlib.patches import Rectangle

xc1 = 2150
yc1 = 1850
w = 80

rect = Rectangle((xc1 - w//2, yc1 - w//2), w, w, fill=False, edgecolor="r")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
im = ax1.imshow(T_SiO2[10])
im.set_clim([0, 1])
ax1.add_patch(rect)
fig.colorbar(im, ax=ax1)
ax1.axis('off')

ax2.plot(wvl, T_SiO2[:, yc1-w//2:yc1+w//2, xc1-w//2:xc1+w//2].mean(axis=(1, 2)), ".-")
ax2.set_ylim([0.95, 1.05])


In [ ]:
n_SiO2 = RIDB().get_material('Fused_Silica_Franta').n(wvl).real

def phase_theory(d, wvl=wvl, n=n_SiO2, sign=+1):
    """Theoretical transmission phase delay of a layer of thickness d, compared to air.

    d   : thickness, in the same units as wvl (e.g. nm)
    wvl : wavelengths, array
    n   : refractive index at each wvl (array of the same shape as wvl, or a scalar)
    sign: +1 or -1, to match the sign convention of the measured phase
    Returns the phase in rad, wrapped to (-π, π] like np.angle output.
    """
    phi = sign * 2*np.pi / wvl * (n - 1) * d
    return np.angle(np.exp(1j * phi))

In [ ]:
from matplotlib.patches import Rectangle

centers     = [(2180, 1870), (2180, 370), (670, 370), (1430, 1120)]   # signal (x, y)
ref_centers = [(2350, 1870), (2350, 370), (820, 370), (1600, 1120)]   # reference (x, y), same order
w = 30
color, ref_color = "r", "c"

def roi_phasor(phi, xc, yc, w):
    """Mean of exp(i*phi) over a w×w box: the circular mean, safe across the ±π wrap."""
    x0, y0 = xc - w//2, yc - w//2
    return np.exp(1j * phi[:, y0:y0+w, x0:x0+w]).mean(axis=(1, 2))

def draw_box(ax, xc, yc, w, label, c):
    x0, y0 = xc - w//2, yc - w//2
    ax.add_patch(Rectangle((x0, y0), w, w, fill=False, edgecolor=c))
    ax.text(x0, y0 + w, label, color=c, ha="right", va="top", fontsize=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
im = ax1.imshow(phi_SiO2[0])
ax1.hlines(600, 0, 2500)
fig.colorbar(im, ax=ax1)
ax1.axis("off")

for i, ((xc, yc), (xr, yr)) in enumerate(zip(centers, ref_centers), start=1):
    draw_box(ax1, xc, yc, w, str(i), color)
    draw_box(ax1, xr, yr, w, str(i), ref_color)

    # phase difference = angle(signal phasor · conj(reference phasor)), wrapped to ±180°
    dphi = np.angle(roi_phasor(phi_SiO2, xc, yc, w) * np.conj(roi_phasor(phi_SiO2, xr, yr, w)))
    ax2.plot(wvl, np.degrees(dphi), ".-", label=str(i))

ax2.set_ylim([-180, 180])
ax2.legend(title="ROI − ref")
ax2.set_xlabel("Wavelength, nm")
ax2.set_ylabel("Δφ, deg")

ax2.plot(wvl, phase_theory(560)*180/np.pi, "k--")

In [ ]:
plt.plot(np.degrees(phi_SiO2[0, 600]))
plt.plot(np.degrees(phi_SiO2[20, 600]))
plt.ylim([-20, 20])
#plt.xlim([500, 2500])

In [ ]:
y_pos = 1130

centers     = [(1810, y_pos), (2080, y_pos)]   # signal (x, y) (770)
ref_centers = [(1950, y_pos), (2230, y_pos)]   # reference (x, y), same order 
w = 30
color, ref_color = "r", "c"

def roi_phasor(phi, xc, yc, w):
    """Mean of exp(i*phi) over a w×w box: the circular mean, safe across the ±π wrap."""
    x0, y0 = xc - w//2, yc - w//2
    return np.exp(1j * phi[:, y0:y0+w, x0:x0+w]).mean(axis=(1, 2))

def draw_box(ax, xc, yc, w, label, c):
    x0, y0 = xc - w//2, yc - w//2
    ax.add_patch(Rectangle((x0, y0), w, w, fill=False, edgecolor=c))
    ax.text(x0, y0 + w, label, color=c, ha="right", va="top", fontsize=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
im = ax1.imshow(phi_SiO2[18])
fig.colorbar(im, ax=ax1)
ax1.axis("off")
ax1.set_title(wvl[18])
ax1.set_xlim([1600, 2300])
ax1.set_ylim([1200, 600])

for i, ((xc, yc), (xr, yr)) in enumerate(zip(centers, ref_centers), start=1):
    draw_box(ax1, xc, yc, w, str(i), color)
    draw_box(ax1, xr, yr, w, str(i), ref_color)

    # phase difference = angle(signal phasor · conj(reference phasor)), wrapped to ±180°
    dphi = np.angle(roi_phasor(phi_SiO2, xc, yc, w) * np.conj(roi_phasor(phi_SiO2, xr, yr, w)))
    ax2.plot(wvl, np.degrees(dphi), ".-", label=str(i))

ax2.set_ylim([-180, 180])
ax2.legend(title="ROI − ref")
ax2.set_xlabel("Wavelength, nm")
ax2.set_ylabel("Δφ, deg")
ax2.grid()

In [ ]:
centers     = [(2200, 370), (1810, 750), (2080, 750), (1810, 1510), (2080, 1510), (1810, 1800), (2200, 1900)]   # signal (x, y) (770)
ref_centers = [(1950, 370), (1950, 750), (2230, 750), (1950, 1510), (2230, 1510), (1950, 1800), (1950, 1900)]   # reference (x, y), same order 
w = 30
color, ref_color = "r", "c"

def roi_phasor(phi, xc, yc, w):
    """Mean of exp(i*phi) over a w×w box: the circular mean, safe across the ±π wrap."""
    x0, y0 = xc - w//2, yc - w//2
    return np.exp(1j * phi[:, y0:y0+w, x0:x0+w]).mean(axis=(1, 2))

def draw_box(ax, xc, yc, w, label, c):
    x0, y0 = xc - w//2, yc - w//2
    ax.add_patch(Rectangle((x0, y0), w, w, fill=False, edgecolor=c))
    ax.text(x0, y0 + w, label, color=c, ha="right", va="top", fontsize=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
im = ax1.imshow(phi_SiO2[18])
fig.colorbar(im, ax=ax1)
ax1.axis("off")
ax1.set_title(wvl[18])
ax1.set_xlim([1650, 2300])
ax1.set_ylim([2050, 200])

for i, ((xc, yc), (xr, yr)) in enumerate(zip(centers, ref_centers), start=1):
    draw_box(ax1, xc, yc, w, str(i), color)
    draw_box(ax1, xr, yr, w, str(i), ref_color)

    # phase difference = angle(signal phasor · conj(reference phasor)), wrapped to ±180°
    dphi = np.angle(roi_phasor(phi_SiO2, xc, yc, w) * np.conj(roi_phasor(phi_SiO2, xr, yr, w)))
    ax2.plot(wvl, np.degrees(np.unwrap((dphi))), ".-", label=str(i))

#ax2.set_ylim([-200, 200])
ax2.legend(title="ROI − ref")
ax2.set_xlabel("Wavelength, nm")
ax2.set_ylabel("Δφ, degrees")
ax2.grid()

# ax2.plot(wvl, np.cos(phase_theory(570)), "k--")
# ax2.plot(wvl, np.cos(phase_theory(610)), "k-.")

In [ ]:
centers     = [(2200, 370), (1810, 750), (2080, 750), (1810, 1510), (2080, 1510), (1810, 1800), (2200, 1900)]   # signal (x, y)
ref_centers = [(1950, 370), (1950, 750), (2230, 750), (1950, 1510), (2230, 1510), (1950, 1800), (1950, 1900)]   # reference (x, y), same order
w = 10
k = 18                      # wavelength index shown in the image
color, ref_color = "r", "c"

def roi_std_of_mean(s, xc, yc, w):
    """Std of the box-averaged value, from per-pixel std s: sqrt(Σσ²)/N (assumes independent pixels)."""
    x0, y0 = xc - w//2, yc - w//2
    box = s[:, y0:y0+w, x0:x0+w]
    return np.sqrt((box**2).sum(axis=(1, 2))) / box[0].size

def draw_box(ax, xc, yc, w, label, c):
    x0, y0 = xc - w//2, yc - w//2
    ax.add_patch(Rectangle((x0, y0), w, w, fill=False, edgecolor=c))
    ax.text(x0, y0 + w, label, color=c, ha="right", va="top", fontsize=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
im = ax1.imshow(phi_SiO2_std[k])
fig.colorbar(im, ax=ax1, label="σ(φ), rad")
ax1.axis("off")
ax1.set_title(f"{wvl[k]:.0f} nm")
ax1.set_xlim([1650, 2300])
ax1.set_ylim([2050, 200])
im.set_clim([0, 0.03])

for i, ((xc, yc), (xr, yr)) in enumerate(zip(centers, ref_centers), start=1):
    draw_box(ax1, xc, yc, w, str(i), color)
    draw_box(ax1, xr, yr, w, str(i), ref_color)

    # σ(Δφ) for Δφ = mean(signal box) − mean(ref box): quadrature sum of the two box-mean stds
    s_dphi = np.sqrt(roi_std_of_mean(phi_SiO2_std, xc, yc, w)**2 +
                     roi_std_of_mean(phi_SiO2_std, xr, yr, w)**2)
    ax2.plot(wvl, np.degrees(s_dphi), ".-", label=str(i))

ax2.legend(title="ROI − ref")
ax2.set_xlabel("Wavelength, nm")
ax2.set_ylabel("σ(Δφ), degrees")
ax2.grid()

In [ ]:
centers     = [(2170, 380)]   # signal (x, y) (770)
ref_centers = [(1840, 370)]   # reference (x, y), same order 
w = 200
color, ref_color = "r", "c"

def roi_phasor(phi, xc, yc, w):
    """Mean of exp(i*phi) over a w×w box: the circular mean, safe across the ±π wrap."""
    x0, y0 = xc - w//2, yc - w//2
    return np.exp(1j * phi[:, y0:y0+w, x0:x0+w]).mean(axis=(1, 2))

def draw_box(ax, xc, yc, w, label, c):
    x0, y0 = xc - w//2, yc - w//2
    ax.add_patch(Rectangle((x0, y0), w, w, fill=False, edgecolor=c))
    ax.text(x0, y0 + w, label, color=c, ha="right", va="top", fontsize=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
im = ax1.imshow(phi_SiO2[60])
fig.colorbar(im, ax=ax1)
ax1.axis("off")
ax1.set_title(wvl[18])
ax1.set_xlim([1650, 2300])
ax1.set_ylim([570, 200])

for i, ((xc, yc), (xr, yr)) in enumerate(zip(centers, ref_centers), start=1):
    draw_box(ax1, xc, yc, w, str(i), color)
    draw_box(ax1, xr, yr, w, str(i), ref_color)

    # phase difference = angle(signal phasor · conj(reference phasor)), wrapped to ±180°
    dphi = np.angle(roi_phasor(phi_SiO2, xc, yc, w) * np.conj(roi_phasor(phi_SiO2, xr, yr, w)))
    ax2.plot(wvl, np.degrees(np.unwrap((dphi))), ".-", label=str(i))

#ax2.set_ylim([-200, 200])
ax2.legend(title="ROI − ref")
ax2.set_xlabel("Wavelength, nm")
ax2.set_ylabel("Δφ, degrees")
ax2.grid()

# ax2.plot(wvl, np.cos(phase_theory(570)), "k--")
# ax2.plot(wvl, np.cos(phase_theory(610)), "k-.")